# 08_08_制約デコーディングを使う_自分で書く

コード：`ch08/blocks/08_08_制約デコーディングを使う_自分で書く.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'transformers==5.22.1', 'tiktoken==0.14.0', 'scikit-learn>=1.5,<2', 'pydantic==2.14.2', 'outlines==1.3.3', 'lancedb==0.41.0', 'pyarrow==26.0.1', 'sentence-transformers==6.2.0']], check=True)


## 本文の掲載コード（参照）


```python
import torch, re

def build_allow_mask(tokenizer, pattern: str, vocab_size: int) -> torch.Tensor:
    """パターンに合うトークンだけを許す（状態を持たない縮小版。Outlines は状態ごとに集合を持つ）"""
    rx = re.compile(pattern)
    # ★ 大きさは語彙表ではなくモデルの出力次元に合わせる（埋め草ぶん大きいことが多い）
    allowed = torch.zeros(vocab_size, dtype=torch.bool)
    for tokstr, tid in tokenizer.get_vocab().items():
        s = tokenizer.convert_tokens_to_string([tokstr])
        if s and rx.fullmatch(s):               # このトークン単体がパターンに合うか
            allowed[tid] = True
    return allowed

@torch.no_grad()
def generate_constrained(model, tok, ids, allowed, max_new=16, temperature=0.7):
    neg = torch.finfo(torch.float32).min
    for _ in range(max_new):
        logits = model(ids).logits[:, -1, :].float()
        logits = logits.masked_fill(~allowed, neg)      # ★ これだけ
        probs  = (logits / temperature).softmax(-1)
        nxt    = torch.multinomial(probs, 1)
        ids    = torch.cat([ids, nxt], dim=-1)
        if nxt.item() == tok.eos_token_id:
            break
    return ids

# 使い方：allowed = build_allow_mask(tok, r"\d+", model.config.vocab_size)
# 注意：全トークンが禁止された状態（＝行き止まり）になると、-inf で埋めた場合は softmax が NaN を返し、
# finfo.min で埋めた場合は一様分布になって意味のない語が選ばれる。
# 実装は必ず「少なくとも 1 つは許可される」ことを文法側で保証している。
# この縮小版は EOS を許可していないので、止まるのは max_new に達したときだけである。
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "08", "--name=constrained", "--skip-docker"], cwd=CODE, check=True)
